# Session 9: Introduction to Bioconductor

**Module 3: Programming for Biological Data**  
**Date:** February 2, 2026 | 18:30 – 21:30  
**Instructor:** Dr. Haogao Gu

---

## Learning Objectives

By the end of this session, you will be able to:
1. Understand the **Bioconductor ecosystem**
2. Install packages using **BiocManager**
3. Work with **Biostrings** for DNA/RNA/Protein sequences
4. Read and manipulate **FASTA** files

In [ ]:
# ============================================
# STANDARD SETUP PROTOCOL (Bioconductor)
# ============================================
options(repos = c(CRAN = "https://cloud.r-project.org"))

# Install BiocManager if needed
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

# Install Biostrings (may take 2-3 minutes first time)
if (!requireNamespace("Biostrings", quietly = TRUE))
  BiocManager::install("Biostrings", update = FALSE, ask = FALSE)

library(Biostrings)

cat("✅ Biostrings loaded successfully!")

---

# Part 1: Bioconductor vs. CRAN

## 40 minutes

---

## 1.1 What is Bioconductor?

**Bioconductor** is a specialized repository for **bioinformatics** packages in R.

| Feature | CRAN | Bioconductor |
|---------|------|-------------|
| Focus | General R packages | Genomics & bioinformatics |
| Install | `install.packages()` | `BiocManager::install()` |
| Releases | Rolling | Twice yearly (Spring/Fall) |
| Packages | ~20,000 | ~2,000 specialized |

## 1.2 Installing Bioconductor Packages

**First time:** Install BiocManager from CRAN

In [ ]:
# Install BiocManager (only needed once)
# install.packages("BiocManager")

# Then install any Bioconductor package:
# BiocManager::install("Biostrings")
# BiocManager::install("GenomicRanges")
# BiocManager::install("sangeranalyseR")

cat("BiocManager version:", as.character(packageVersion("BiocManager")))

## 1.3 Key Bioconductor Packages for MLTs

| Package | Purpose |
|---------|--------|
| **Biostrings** | DNA, RNA, amino acid strings |
| **sangeranalyseR** | Sanger sequencing analysis |
| **GenomicRanges** | Genome coordinate operations |
| **msa** | Multiple sequence alignment |
| **ggtree** | Phylogenetic tree visualization |

---

# Part 2: Biostrings

## 40 minutes

---

## 2.1 Why Not Regular Strings?

R character strings work, but Biostrings are **optimized** for sequences:

- Memory efficient for long sequences
- Specialized functions (complement, translate, etc.)
- Validation (only valid nucleotides allowed)

In [ ]:
# Regular R string (works but limited)
dna_string <- "ATGCGATCGATCG"

# Biostrings DNAString (specialized)
dna_seq <- DNAString("ATGCGATCGATCG")

print(dna_seq)
cat("\nLength:", length(dna_seq), "bp")

## 2.2 DNAString and DNAStringSet

In [ ]:
# Single sequence
single_seq <- DNAString("ATGATGATGATG")

# Multiple sequences (like a FASTA file)
multi_seq <- DNAStringSet(c(
  "Sample1" = "ATGATGATGATG",
  "Sample2" = "ATGCTGCTGCTG",
  "Sample3" = "ATGTTTTTTTTT"
))

print(multi_seq)

## 2.3 Sequence Operations

In [ ]:
# Create a sample sequence
my_seq <- DNAString("ATGAAACCCGGGTTTTAA")

# Complement (A↔T, G↔C)
cat("Original:   ", as.character(my_seq), "\n")
cat("Complement: ", as.character(complement(my_seq)), "\n")

# Reverse complement (what you'd see on opposite strand)
cat("RevComp:    ", as.character(reverseComplement(my_seq)), "\n")

In [ ]:
# Translate DNA to protein (in-silico translation)
protein <- translate(my_seq)

cat("DNA:     ", as.character(my_seq), "\n")
cat("Protein: ", as.character(protein), "\n")
cat("\nAmino acids:", length(protein))

## 2.4 Nucleotide Frequencies and GC Content

In [ ]:
# Count nucleotides
my_seq <- DNAString("ATGCATGCATGCGGGCCCAAATTT")

freq <- alphabetFrequency(my_seq, baseOnly = TRUE)
print(freq)

# Calculate GC content
gc_count <- freq["G"] + freq["C"]
total <- sum(freq[c("A", "C", "G", "T")])
gc_content <- gc_count / total * 100

cat("\nGC Content:", round(gc_content, 1), "%")

In [ ]:
# Shortcut: letterFrequency
gc <- letterFrequency(my_seq, letters = c("G", "C"), as.prob = TRUE)
cat("GC Content:", round(sum(gc) * 100, 1), "%")

## 2.5 Reading FASTA Files

In [ ]:
# Create demo FASTA content
fasta_content <- ">Wuhan-Hu-1
ATGTTTGTTTTTCTTGTTTTATTGCCACTAGTCTCTAGTCAGTGTGTTAATCTTACAACC
>Alpha_B.1.1.7
ATGTTTGTTTTTCTTGTTTTATTGCCACTAGTCTCTAGTCAGTGTGTTAATCTTACAACC
>Delta_B.1.617.2
ATGTTTGTTTTTCTTGTTTTATTGCCACTAGTCTCTAGTCAGTGTGTTAATCTTACAACC
>Omicron_BA.1
ATGTTTGTTTTTCTTGTTTTATTGCCACTAGTCTCTAGTCAGTGTGTTAATCTTACAACC"

# Write to temp file
writeLines(fasta_content, "temp_sequences.fasta")

# Read FASTA file
sequences <- readDNAStringSet("temp_sequences.fasta")

print(sequences)

In [ ]:
# Access individual sequences
cat("Number of sequences:", length(sequences), "\n")
cat("Sequence names:\n")
print(names(sequences))

# Access first sequence
cat("\nFirst sequence:")
print(sequences[[1]])

## 2.6 File Format Summary

| Format | Extension | Contains | R Function |
|--------|-----------|----------|------------|
| FASTA | .fasta, .fa | Sequence only | `readDNAStringSet()` |
| FASTQ | .fastq, .fq | Sequence + Quality | `readDNAStringSet(format="fastq")` |
| AB1 | .ab1 | Chromatogram trace | `sangeranalyseR` |

## 2.7 FASTA vs FASTQ

**FASTA:**
```
>Sequence_Name
ATGCATGCATGC...
```

**FASTQ:**
```
@Sequence_Name
ATGCATGCATGC...
+
IIIIIIIIIIII...  (quality scores)
```

## 2.8 Working with Multiple Sequences

In [ ]:
# Calculate GC content for all sequences
gc_all <- letterFrequency(sequences, letters = c("G", "C"), as.prob = TRUE)
gc_percent <- rowSums(gc_all) * 100

result <- data.frame(
  Variant = names(sequences),
  Length = width(sequences),
  GC_Content = round(gc_percent, 1)
)

print(result)

In [ ]:
# Translate all sequences
proteins <- translate(sequences)
print(proteins)

---

# Key Takeaways

1. **Bioconductor** = specialized bioinformatics packages

2. **BiocManager::install()** = how to install Bioconductor packages

3. **DNAString / DNAStringSet** = efficient sequence objects

4. Key functions:
   - `complement()`, `reverseComplement()`
   - `translate()` — DNA to protein
   - `letterFrequency()` — nucleotide counts
   - `readDNAStringSet()` — read FASTA

5. **FASTA** = sequence, **FASTQ** = sequence + quality

---

## Key Functions

```r
library(Biostrings)
seq <- DNAString("ATGC...")
seqs <- readDNAStringSet("file.fasta")
complement(seq)
reverseComplement(seq)
translate(seq)
letterFrequency(seq, letters = c("G","C"))
```

---

## Now proceed to Tutorial 9! 🧬